In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import os

df = pd.read_csv(os.path.join(path, 'Q3_data.csv'))
df.shape

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

print(f"\nAfter: {df.isnull().sum().sum()} missing values")

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()
print(f"Duplicates: {duplicates}")
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f"Removed {duplicates} duplicates")

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'target' in categorical_cols:
    categorical_cols.remove('target')

print(f"Categorical: {categorical_cols}")

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    print(f"Encoded: {col}")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

feature_cols = [col for col in df.columns if col != 'target']
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])
print("Scaled with StandardScaler")

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns

print("Target distribution:")
print(df['Target'].value_counts())
print("\nPercentages:")
print(df['Target'].value_counts(normalize=True) * 100)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Target')
plt.title('Target Distribution')
plt.show()

class_0 = (df['Target'] == 0).sum() / len(df) * 100
if class_0 > 70 or class_0 < 30:
    print("Dataset is IMBALANCED")
else:
    print("Dataset is balanced")

In [ ]:
# Task 1: Write your code here:
X = df.drop('Target', axis=1)
y = df['Target']
print(f"X: {X.shape}")
print(f"y: {y.shape}")

In [ ]:
!pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(iterations=100, random_state=42, verbose=0)

f1_scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred, average='weighted')
    f1_scores.append(f1)
    print(f"Fold {fold}: F1 = {f1:.4f}")

print(f"\nAverage F1: {np.mean(f1_scores):.4f}")

In [ ]:
# Task 1: Write your code here:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(importances['feature'][:20], importances['importance'][:20])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 20 Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
golden_feature = importances.iloc[0]['feature']
golden_importance = importances.iloc[0]['importance']

print(f"Golden Feature: {golden_feature}")
print(f"Importance: {golden_importance:.4f}")

In [ ]:
# Task Bonus: Write your code here:
X_golden = df[[golden_feature]]
y = df['target']

model_golden = CatBoostClassifier(iterations=100, random_state=42, verbose=0)
f1_golden = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_golden, y), 1):
    X_train = X_golden.iloc[train_idx]
    X_test = X_golden.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model_golden.fit(X_train, y_train)
    y_pred = model_golden.predict(X_test)

    f1 = f1_score(y_test, y_pred, average='weighted')
    f1_golden.append(f1)
    print(f"Fold {fold}: F1 = {f1:.4f}")

print(f"\nFull Model: {np.mean(f1_scores):.4f}")
print(f"Golden Only: {np.mean(f1_golden):.4f}")
print(f"Performance: {(np.mean(f1_golden)/np.mean(f1_scores)*100):.1f}%")